# Pre-extract DINOv2 features

Chạy notebook này **một lần** để trích và lưu DINOv2 CLS features cho từng dataset.
`run_al.ipynb` sẽ tự động dùng lại các file này (khớp dataset + seed + backbone) nên không phải trích lại mỗi lần chạy.

**Lưu ý về seed:** dataset dạng ImageFolder (histoset, skintissue) được split bằng seeded generator, nên feature chỉ dùng lại được khi SEED khớp. Trích sẵn cho đúng seed sẽ dùng khi chạy AL.

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "tiendung"
CODAPATH = Path("/kaggle/working/codapath")
if (CODAPATH / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(CODAPATH), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(CODAPATH), "pull", "--ff-only", "origin", REPO_BRANCH])
elif CODAPATH.exists():
    raise RuntimeError(f"{CODAPATH} exists but is not a Git repository")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(CODAPATH)])
actual_branch = subprocess.check_output(["git", "-C", str(CODAPATH), "branch", "--show-current"], text=True).strip()
assert actual_branch == REPO_BRANCH, (actual_branch, REPO_BRANCH)
print("repo:", CODAPATH, "| branch:", actual_branch)

In [ ]:
%cd /kaggle/working/codapath
CODAPATH = "/kaggle/working/codapath"

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

In [ ]:
# Datasets to extract. Must match the checkpoint/save_dir names used by run_al.
DATASETS = ["pathmnist", "histoset", "skintissue"]

# Extract for every seed you plan to run AL with (ImageFolder split depends on seed).
SEEDS = [42]

# Where to write the .npy caches. On Kaggle this lives under /kaggle/working and
# is saved as notebook output; add it as an input dataset to run_al and set the
# matching FEATURE_DIR there.
FEATURE_DIR = "/kaggle/working/features"

In [ ]:
import os
from huggingface_hub import snapshot_download

# DINOv2 is public; Kaggle Internet must be enabled.
print("Downloading facebook/dinov2-base...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import sys

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if CODAPATH not in sys.path:
    sys.path.append(CODAPATH)

In [ ]:
import yaml
import torch

from load_data import get_data_loaders, get_sample_ids, sample_order_fingerprint
from model import get_or_extract_features

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])
PATHMNIST_PATH  = str(DATA_ROOT / "pathmnist_224.npz")
HISTOSET_PATH   = str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14")
SKINTISSUE_PATH = str(DATA_ROOT / "SkinTissue/SkinTissue/tiles")

DATA_DICT = {
    "pathmnist":  PATHMNIST_PATH,
    "histoset":   HISTOSET_PATH,
    "skintissue": SKINTISSUE_PATH,
}

In [ ]:
CONFIG_PATH = "config/config.yaml"

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DEVICE   = torch.device(config["device"])
VIT_NAME = config.get("models", {}).get("vit", "facebook/dinov2-base")
assert torch.cuda.is_available(), "Attach a Kaggle GPU before extraction"
missing_paths = {ds: DATA_DICT[ds] for ds in DATASETS if not Path(DATA_DICT[ds]).exists()}
assert not missing_paths, f"Missing Kaggle inputs: {missing_paths}"
Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE} | backbone: {VIT_NAME} | cache: {FEATURE_DIR}")

In [ ]:
for seed in SEEDS:
    for ds in DATASETS:
        print("=" * 60)
        print(f"Dataset: {ds} | seed: {seed}")
        train_loader, test_loader, _ = get_data_loaders(DATA_DICT[ds], seed, verbose=True)
        train_features, test_features = get_or_extract_features(
            train_loader, test_loader, ds, seed, VIT_NAME, DEVICE, cache_dir=FEATURE_DIR,
            train_fingerprint=sample_order_fingerprint(get_sample_ids(train_loader.dataset)),
            test_fingerprint=sample_order_fingerprint(get_sample_ids(test_loader.dataset)),
        )
        print(f"  train {train_features.shape} | test {test_features.shape}")

print("\nDone. Cached files:")
for f in sorted(os.listdir(FEATURE_DIR)):
    print("  ", os.path.join(FEATURE_DIR, f))